# Downloading gravity data from the Colorado Geoid Experiment

Accessed from:
* https://geodesy.noaa.gov/GEOID/research/co-cm-experiment/
* https://www.ngs.noaa.gov/GRAV-D/data_ms05.shtml
* https://www.isgeoid.polimi.it/Projects/colorado_experiment.html

Metadata: https://geodesy.noaa.gov/GEOID/research/co-cm-experiment/readme.pdf

In [ ]:
import pandas as pd
import pooch

import airbornegeo

## load data

In [ ]:
# fname = pooch.retrieve(
#     url="https://geodesy.noaa.gov/GEOID/research/co-cm-experiment/GRAVD_ms05_median_debiased_1hz.txt",
#     progressbar=True,
#     path=f"{pooch.os_cache('airbornegeo')}",
#     known_hash='08e60788a425d6937d3eb27e370ff61967ea4e58c285d644c82f99b15e904815',
# )

# data_df = pd.read_csv(
#     fname,
#     header=0,
# )
# data_df = data_df.rename(columns={
#     'time(YYYYMMDDHHMMSS.SSS)': 'time',
#     'lat(deg)': 'lat',
#     'lon(deg)': 'lon',
#     'h(m)': 'height',
#     'g (mGal)': 'gravity',
# })

# data_df['time'] = pd.to_datetime(data_df["time"])
# data_df["unixtime"] = data_df["time"].apply(lambda x: x.timestamp())

# # sort by line then time and reset index
# data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)

# data_df

In [ ]:
fname = pooch.retrieve(
    url="https://geodesy.noaa.gov/pub/grav-d/MS05/NGS_GRAVD_Block_MS05_BETA2.zip",
    progressbar=True,
    path=f"{pooch.os_cache('airbornegeo')}",
    known_hash="ef3fdaa0dde7d3cc2ad26e318cf0ce297762ca8d5b0fab74ef13faef496f27c0",
    processor=pooch.Unzip(members=["NGS_GRAVD_Block_MS05_Gravity_Data_BETA2.txt"]),
)[0]
fname

In [ ]:
data_df = pd.read_csv(
    fname,
    sep=r"\s+",
    header=None,
    names=[
        "flight",
        "time",
        "latitude",
        "longitude",
        "height_geoidal",
        "g_obs",
    ],
)

data_df["time"] = pd.to_datetime(data_df["time"])
data_df["unixtime"] = data_df["time"].apply(lambda x: x.timestamp())

# convert line ID's from strings to unique floats
data_df["line"] = airbornegeo.unique_line_id(
    data_df,
    line_col_name="flight",
)

# sort by line then time and reset index
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)

# reproject to cartesian coordiantes
data_df["easting"], data_df["northing"] = airbornegeo.reproject(
    data_df.longitude,
    data_df.latitude,
    input_crs="epsg:4326",
    output_crs="epsg:32613",  # UTM Zone 13 N
)
data_df.head()

In [ ]:
data_df.describe()

In [ ]:
df = data_df[::10]
ax = df.plot.scatter(
    "easting",
    "northing",
    c="line",
    s=0.02,
    cmap="rainbow",
    title="Lines",
)
ax.set_aspect("equal")

In [ ]:
df = data_df[::10]
ax = df.plot.scatter(
    "easting",
    "northing",
    c="g_obs",
    s=0.2,
    title="Gravity",
)
ax.set_aspect("equal")

In [ ]:
df = data_df[::10]
ax = df.plot.scatter(
    "easting",
    "northing",
    c="height_geoidal",
    s=0.2,
    title="Gravity",
)
ax.set_aspect("equal")